In [ ]:
import sys
sys.path.insert(0, '../lib')

import json
import os
import re
import functools

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import common_data

In [2]:
pd.options.display.max_columns = 300
pd.options.display.max_rows = 200
pd.options.display.max_colwidth = 10000

# Binning EHR dataframe for EHRformer pretraining

We want transformer to unmask our different EHR measurements. To handle different measurements and make it a transformer problem, we bin each measurement into $X$ bins, and add 1 bin for indication of missing data.

In [ ]:
data = pd.read_csv(common_data.CLINICAL, index_col=0)

To preserve many things from development, let's reorder the rows based on internal patient IDs

In [106]:
data.sort_values(['patient', 'ICU_stay', 'ICU_day'], inplace=True)

In [ ]:
labels = pd.read_csv(common_data.CLINICAL_LABELS, index_col=0)

In [ ]:
to_copy = [
    'day_of_hospitalization',
    'week_of_hospitalization',
    'days_on_ventilator',
    'mean_nat_score_until_yesterday',
]
for col in to_copy:
    data[col] = labels[col]

In [ ]:
data_columns_numeric = [
    'Temperature',
    'Heart_rate',
    'Systolic_blood_pressure',
    'Diastolic_blood_pressure',
    'Mean_arterial_pressure',
    'Norepinephrine_rate',
    'Respiratory_rate',
    'Oxygen_saturation',
    'Urine_output',
    'PEEP',
    'FiO2',
    'Plateau_Pressure',
    'Lung_Compliance',
    'Minute_Ventilation',
    'PEEP_changes',
    'Respiratory_rate_changes',
    'FiO2_changes',
    'ABG_pH',
    'ABG_PaCO2',
    'ABG_PaO2',
    'PaO2FIO2_ratio',
    'WBC_count',
    'Lymphocytes',
    'Neutrophils',
    'Hemoglobin',
    'Platelets',
    'Bicarbonate',
    'Creatinine',
    'Albumin',
    'Bilirubin',
    'LDH',
    'Lactic_acid',
    'Procalcitonin',
    'Steroid_dose',
    'Age',
    'BMI',
    'ICU_stay',
] + to_copy

## Bin numerical columns

Using pandas `qcut`. Sometimes, however, the one value of the data will have more than one quartile, so then we won't get 4 bins, but less. Let's print those cases.

In [110]:
cols = []
problem_cols = []
def bin_column(column, n_bins=4):
    try:
        result, bins = pd.qcut(column, q=n_bins, labels=False, retbins=True)#, duplicates='drop')
    except ValueError:
        print(f'Problem binning {column.name}, do it manually')
        problem_cols.append(column.name)
        return
    result = pd.get_dummies(result)
    result['is_na'] = column.isna().astype(int)
    result.columns = pd.MultiIndex.from_product([[column.name], [f'bin{str(col).split(".")[0]}' for col in result.columns]])
    cols.append(result)

data[data_columns_numeric].apply(bin_column, axis=0)

Problem binning PEEP_changes, do it manually
Problem binning Respiratory_rate_changes, do it manually
Problem binning FiO2_changes, do it manually
Problem binning Steroid_dose, do it manually
Problem binning ICU_stay, do it manually
Problem binning week_of_hospitalization, do it manually


Temperature                       None
Heart_rate                        None
Systolic_blood_pressure           None
Diastolic_blood_pressure          None
Mean_arterial_pressure            None
Norepinephrine_rate               None
Respiratory_rate                  None
Oxygen_saturation                 None
Urine_output                      None
PEEP                              None
FiO2                              None
Plateau_Pressure                  None
Lung_Compliance                   None
Minute_Ventilation                None
PEEP_changes                      None
Respiratory_rate_changes          None
FiO2_changes                      None
ABG_pH                            None
ABG_PaCO2                         None
ABG_PaO2                          None
PaO2FIO2_ratio                    None
WBC_count                         None
Lymphocytes                       None
Neutrophils                       None
Hemoglobin                        None
Platelets                

## Process problematic columns manually

In [111]:
BINS = [0, 1, 2, 3]

### PEEP_changes

In [112]:
data.PEEP_changes.value_counts()

1.0     8179
2.0     2289
3.0     1293
4.0      502
5.0      151
6.0       50
7.0       22
8.0        4
10.0       1
Name: PEEP_changes, dtype: int64

In [113]:
PEEP_changes = data['PEEP_changes']
conditions = [
    (PEEP_changes == 1),
    ((PEEP_changes > 1) & (PEEP_changes <= 3)),
    ((PEEP_changes > 3) & (PEEP_changes <= 4)),
    (PEEP_changes > 4)
]

data['PEEP_changes'] = np.select(conditions, BINS, np.nan)

In [114]:
data.PEEP_changes.value_counts()

0.0    8179
1.0    3582
2.0     502
3.0     228
Name: PEEP_changes, dtype: int64

### Respiratory rate changes

In [115]:
data.Respiratory_rate_changes.value_counts()

1.0    6743
2.0    2192
3.0    1879
4.0    1130
5.0     379
6.0     116
7.0      38
8.0      14
Name: Respiratory_rate_changes, dtype: int64

In [116]:
Respiratory_rate_changes = data['Respiratory_rate_changes']
conditions = [
    (Respiratory_rate_changes == 1),
    ((Respiratory_rate_changes > 1) & (Respiratory_rate_changes <= 3)),
    ((Respiratory_rate_changes > 3) & (Respiratory_rate_changes <= 4)),
    (Respiratory_rate_changes > 4)
]

data['Respiratory_rate_changes'] = np.select(conditions, BINS, np.nan)

In [117]:
data.Respiratory_rate_changes.value_counts()

0.0    6743
1.0    4071
2.0    1130
3.0     547
Name: Respiratory_rate_changes, dtype: int64

### FiO2 changes

In [118]:
data.FiO2_changes.value_counts()

1.0    6351
2.0    3134
3.0    2019
4.0     744
5.0     185
6.0      46
7.0      10
8.0       2
Name: FiO2_changes, dtype: int64

In [119]:
FiO2_changes = data['FiO2_changes']
conditions = [
    (FiO2_changes == 1),
    ((FiO2_changes > 1) & (FiO2_changes <= 3)),
    ((FiO2_changes > 3) & (FiO2_changes <= 4)),
    (FiO2_changes > 4)
]

data['FiO2_changes'] = np.select(conditions, BINS, np.nan)

In [120]:
data.FiO2_changes.value_counts()

0.0    6351
1.0    5153
2.0     744
3.0     243
Name: FiO2_changes, dtype: int64

## Steroid dose

In [121]:
data.Steroid_dose.value_counts().head()

0.0      9326
150.0    1276
200.0     982
20.0      538
100.0     472
Name: Steroid_dose, dtype: int64

Too many zeroes

In [122]:
pd.qcut(data.Steroid_dose, q=4, labels=False, retbins=True)#, duplicates='drop')

ValueError: Bin edges must be unique: array([    0.,     0.,     0.,   100., 10000.]).
You can drop duplicate edges by setting the 'duplicates' kwarg

They form first 2 bins. Let's make them form 1 bin and bin the rest in 3 bins

In [123]:
idx = data.Steroid_dose.gt(0)
Steroid_dose, bins = pd.qcut(data.Steroid_dose[idx], q=3, labels=False, retbins=True)

In [124]:
bins[0], bins[1], bins[2], bins[3]

(5.0, 100.0, 200.0, 10000.0)

In [125]:
Steroid_dose += 1

In [126]:
data.loc[idx, 'Steroid_dose'] = Steroid_dose

In [127]:
data.Steroid_dose.value_counts()

0.0    9326
2.0    2620
1.0    2247
3.0    1094
Name: Steroid_dose, dtype: int64

### ICU_stay

In [128]:
data.ICU_stay.value_counts()

1    12878
2     1615
3      425
4      152
5      143
6       64
8        7
7        3
Name: ICU_stay, dtype: int64

In [129]:
ICU_stay = data['ICU_stay']
conditions = [
    (ICU_stay == 1),
    ((ICU_stay > 1) & (ICU_stay <= 2)),
    ((ICU_stay > 2) & (ICU_stay <= 4)),
    (ICU_stay > 4)
]

data['ICU_stay'] = np.select(conditions, BINS, np.nan)

In [130]:
data.ICU_stay.value_counts()

0.0    12878
1.0     1615
2.0      577
3.0      217
Name: ICU_stay, dtype: int64

### Week of hospitalization

In [131]:
data.week_of_hospitalization.value_counts()

1     3840
2     3180
3     2205
4     1564
5     1166
6      810
7      540
8      391
9      302
10     196
11     161
12     121
13      90
15      90
14      88
16      86
17      67
18      50
22      40
19      39
21      35
20      35
23      31
24      21
25      21
26      21
27      21
28      17
29      14
30       9
31       7
32       7
33       7
34       7
35       7
36       1
Name: week_of_hospitalization, dtype: int64

In [132]:
week_of_hospitalization = data['week_of_hospitalization']
conditions = [
    (week_of_hospitalization == 1),
    (week_of_hospitalization == 2),
    ((week_of_hospitalization > 2) & (week_of_hospitalization <= 6)),
    (week_of_hospitalization > 6)
]

data['week_of_hospitalization'] = np.select(conditions, BINS, np.nan)

In [133]:
data.week_of_hospitalization.value_counts()

2.0    5745
0.0    3840
1.0    3180
3.0    2522
Name: week_of_hospitalization, dtype: int64

## Bin categorical columns manually

In [ ]:
# separate numeric from categorical
categorical_features = [
    'Gender',
    'Intubation_flag',
    'Hemodialysis_flag',
    'CRRT_flag',
    'GCS_eye_opening',
    'GCS_motor_response',
    'GCS_verbal_response',
    'RASS_score',
    'Immunocompromised_flag'
]

In [135]:
# change gender so that male = 0 and female = 3 (opposite bins)

d = {'Male': 0, 'Female': 3}
data['Gender'] = data['Gender'].map(d).fillna(data['Gender'])

In [136]:
# change all other binary variables
# bin RASS score

ECMO_flag = data['ECMO_flag']

conditions = [
      (ECMO_flag == 0)
    , (ECMO_flag == 1)
]

# List of values to return
choices  = [
      0
    , 3
]

# create a new column in the DF based on the conditions
data["ECMO_flag"] = np.select(conditions, choices, np.nan)

In [137]:
# change all other binary variables
# bin RASS score

Intubation_flag = data['Intubation_flag']

conditions = [
      (Intubation_flag == 0)
    , (Intubation_flag == 1)
]

# List of values to return
choices  = [
      0
    , 3
]

# create a new column in the DF based on the conditions
data["Intubation_flag"] = np.select(conditions, choices, np.nan)

In [138]:
# change all other binary variables
# bin RASS score

CRRT_flag = data['CRRT_flag']

conditions = [
      (CRRT_flag == 0)
    , (CRRT_flag == 1)
]

# List of values to return
choices  = [
      0
    , 3
]

# create a new column in the DF based on the conditions
data["CRRT_flag"] = np.select(conditions, choices, np.nan)

In [139]:
Hemodialysis_flag = data['Hemodialysis_flag']

conditions = [
      (Hemodialysis_flag == 0)
    , (Hemodialysis_flag == 1)
]

# List of values to return
choices  = [
      0
    , 3
]

# create a new column in the DF based on the conditions
data["Hemodialysis_flag"] = np.select(conditions, choices, np.nan)

In [140]:
Immunocompromised_flag = data['Immunocompromised_flag']

conditions = [
      (Immunocompromised_flag == 0)
    , (Immunocompromised_flag == 1)
]

# List of values to return
choices  = [
      0
    , 3
]

# create a new column in the DF based on the conditions
data["Immunocompromised_flag"] = np.select(conditions, choices, np.nan)

In [141]:
# bin RASS score

RASS_score = data['RASS_score']

conditions = [
      (RASS_score < -3)
    , ((RASS_score < -1) & (RASS_score >= -3))
    , ((RASS_score < 2) & (RASS_score >= -1))
    , (RASS_score >= 2)
]

# List of values to return
choices  = [
      0
    , 1
    , 2
    , 3
]

# create a new column in the DF based on the conditions
data["RASS_score"] = np.select(conditions, choices, np.nan)

In [142]:
# bin GCS scores

GCS_verbal_response = data['GCS_verbal_response']

conditions = [
      (GCS_verbal_response < 2) # <2, no response
    , ((GCS_verbal_response < 4) & (GCS_verbal_response >= 2))
    , ((GCS_verbal_response < 5) & (GCS_verbal_response >= 4))
    , (GCS_verbal_response >= 5)
]

# List of values to return
choices  = [
      0
    , 1
    , 2
    , 3
]

# create a new column in the DF based on the conditions
data["GCS_verbal_response"] = np.select(conditions, choices, np.nan)

In [143]:
# bin GCS scores

GCS_motor_response = data['GCS_motor_response']

conditions = [
      (GCS_motor_response < 2) # <2, no response
    , ((GCS_motor_response < 4) & (GCS_motor_response >= 2))
    , ((GCS_motor_response < 6) & (GCS_motor_response >= 4))
    , (GCS_motor_response >= 6)
]

# List of values to return
choices  = [
      0
    , 1
    , 2
    , 3
]

# create a new column in the DF based on the conditions
data["GCS_motor_response"] = np.select(conditions, choices, np.nan)

In [144]:
GCS_eye_opening = data['GCS_eye_opening']

conditions = [
      (GCS_eye_opening == 1) # <2, no response
    , (GCS_eye_opening == 2)
    , (GCS_eye_opening == 3)
    , (GCS_eye_opening == 4)
]

# List of values to return
choices  = [
      0
    , 1
    , 2
    , 3
]

# create a new column in the DF based on the conditions
data["GCS_eye_opening"] = np.select(conditions, choices, np.nan)

In [145]:
# one hot encoding function for categorical
cols_cat = []
def one_hot(column):
    result = pd.get_dummies(column) # this does the one hot encoding
    if column.nunique() < 4:
        result.insert(loc = 1, column = '1.0', value = 0)
        result.insert(loc = 2, column = '2.0', value = 0)
    result['is_na'] = column.isna().astype(int)
    result.columns = pd.MultiIndex.from_product([[column.name], [f'bin{str(col).split(".")[0]}' for col in result.columns]])
    cols_cat.append(result)

data[categorical_features + problem_cols].apply(one_hot, axis=0)

Gender                      None
Intubation_flag             None
Hemodialysis_flag           None
CRRT_flag                   None
GCS_eye_opening             None
GCS_motor_response          None
GCS_verbal_response         None
RASS_score                  None
Immunocompromised_flag      None
PEEP_changes                None
Respiratory_rate_changes    None
FiO2_changes                None
Steroid_dose                None
ICU_stay                    None
week_of_hospitalization     None
dtype: object

In [146]:
cols = cols + cols_cat

In [147]:
len(cols)

50

In [148]:
binned = pd.concat(cols, axis=1)

In [ ]:
binned.to_pickle(common_data.EHRFORMER_BINS)